# Celda 1 — Instalar librerías

In [ ]:
!pip install -q gliner openpyxl tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 94.1 MB/s eta 0:00:00


# Celda 2 — Imports y configuración

In [ ]:
import re
import json
import math
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from gliner import GLiNER

In [ ]:
# =========================
# CONFIGURACIÓN GENERAL
# =========================

INPUT_CSV = Path("/content/embargos_limpieza_avanzada_80_mejores.csv")

OUTPUT_DIR = Path("/content/salida_extraccion_embargos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "fastino/gliner2-privacy-filter-PII-multi"

# En tu CSV actual existe texto_limpieza_avanzada.
# Dejo fallback por si cambia el nombre de columna.
TEXT_COL_PRIORITY = [
    "texto_limpieza_avanzada",
    "texto_limpio",
    "texto_ocr",
    "texto_base",
    "texto_markdown"
]

MAX_EMBARGOS = 80

# GLiNER
GLINER_LABELS = ["person"]
GLINER_THRESHOLD = 0.35

# Chunking para documentos largos
CHUNK_SIZE = 2800
CHUNK_OVERLAP = 400

# Para evitar falsos positivos como montos muy chicos o pedazos de OCR
MIN_MONTO_CANDIDATO = 1000

# Celda 3 — Cargar CSV

In [ ]:
# Si el archivo no está en /content, Colab va a pedir que lo subas manualmente.
if not INPUT_CSV.exists():
    from google.colab import files
    uploaded = files.upload()
    INPUT_CSV = Path(next(iter(uploaded.keys())))

df = pd.read_csv(INPUT_CSV)

# Trabajamos con los 80 embargos seleccionados
df = df.head(MAX_EMBARGOS).copy()

# Detectar columna de texto
text_col = None
for col in TEXT_COL_PRIORITY:
    if col in df.columns:
        text_col = col
        break

if text_col is None:
    raise ValueError(f"No encontré ninguna columna de texto. Columnas disponibles: {df.columns.tolist()}")

id_col = "id" if "id" in df.columns else None
nombre_col = "nombre" if "nombre" in df.columns else None
clasificacion_col = "clasificacion" if "clasificacion" in df.columns else None

print("Filas cargadas:", len(df))
print("Columna de texto usada:", text_col)
print("Columna ID:", id_col)
print("Columna nombre:", nombre_col)

Saving embargos_limpieza_avanzada_80_mejores.csv to embargos_limpieza_avanzada_80_mejores.csv
Filas cargadas: 80
Columna de texto usada: texto_limpieza_avanzada
Columna ID: id
Columna nombre: nombre


# Celda 4 — Funciones auxiliares

Estas funciones limpian valores, extraen contexto y normalizan números argentinos.

In [ ]:
def safe_str(x):
    if x is None:
        return ""
    try:
        if isinstance(x, float) and math.isnan(x):
            return ""
    except Exception:
        pass
    return str(x)


def clean_context(s):
    s = safe_str(s).replace("\n", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def get_context(text, start, end, window=220):
    a = max(0, start - window)
    b = min(len(text), end + window)
    return clean_context(text[a:b])


def normalize_digits(value):
    return re.sub(r"\D", "", safe_str(value))


def normalize_money_ar(value):
    """
    Convierte montos argentinos:
    $108.332,50 -> 108332.50
    $81.249 -> 81249.00
    """
    s = safe_str(value)
    s = s.replace("$", "")
    s = re.sub(r"(?i)\bARS\b", "", s)
    s = re.sub(r"(?i)\bpesos?\b", "", s)
    s = re.sub(r"[^\d,\.]", "", s).strip()

    if not s:
        return None

    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")
    elif "." in s:
        parts = s.split(".")
        if len(parts) > 2:
            s = "".join(parts)
        elif len(parts[-1]) == 3:
            s = "".join(parts)

    try:
        return float(s)
    except ValueError:
        return None


def entity_dict(tipo, valor, valor_normalizado, metodo, start, end, text, confianza=None, extra=None):
    d = {
        "tipo": tipo,
        "valor": clean_context(valor),
        "valor_normalizado": valor_normalizado,
        "metodo": metodo,
        "span_inicio": int(start) if start is not None else None,
        "span_fin": int(end) if end is not None else None,
        "contexto": get_context(text, int(start), int(end)) if start is not None and end is not None else "",
        "confianza": float(confianza) if confianza is not None else None,
    }

    if extra:
        d.update(extra)

    return d


def chunk_text(text, chunk_size=2800, overlap=400):
    text = safe_str(text)
    n = len(text)

    if n <= chunk_size:
        yield 0, text
        return

    start = 0
    while start < n:
        end = min(n, start + chunk_size)
        yield start, text[start:end]

        if end == n:
            break

        start = max(0, end - overlap)


def deduplicate_entities(entities):
    """
    Elimina duplicados exactos.
    Sirve especialmente porque GLiNER procesa el texto por chunks con solapamiento.
    """
    result = []
    seen = set()

    for e in sorted(entities, key=lambda x: (x.get("span_inicio") or 0, x.get("span_fin") or 0)):
        norm = safe_str(e.get("valor_normalizado", e.get("valor", ""))).lower().strip()
        key = (
            e.get("tipo"),
            norm,
            e.get("span_inicio"),
            e.get("span_fin")
        )

        if key not in seen:
            seen.add(key)
            result.append(e)

    return result


def to_jsonable(obj):
    """
    Convierte numpy/pandas types a tipos serializables por JSON.
    """
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(to_jsonable(v) for v in obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if pd.isna(obj) if not isinstance(obj, (list, dict, tuple)) else False:
        return None
    return obj

# Celda 5 — Regex para DNI, CUIT, CUIL, CBU, CVU, alias y montos

In [ ]:
# =========================
# REGEX ARGENTINA
# =========================

PAT_CBU_CVU = re.compile(
    r"\b(?P<label>CBU|CVU)\s*(?:N[°ºo]?\s*)?[:\-]?\s*(?P<value>(?:\d[\s\.\-]*){22})\b",
    re.IGNORECASE
)

PAT_CUIL_CUIT = re.compile(
    r"\b(?P<label>CUIT|CUIL|C\.U\.I\.T\.|C\.U\.I\.L\.)\s*(?:N[°ºo]?\s*)?[:\-]?\s*(?P<value>\d{2}[\s\.\-]?\d{7,8}[\s\.\-]?\d)\b",
    re.IGNORECASE
)

PAT_DNI = re.compile(
    r"\b(?P<label>DNI|D\.N\.I\.|DOCUMENTO|DOC\.)\s*(?:N[°ºo]?\s*)?[:\-]?\s*(?P<value>\d{1,2}[\.\s]?\d{3}[\.\s]?\d{3}|\d{7,8})\b",
    re.IGNORECASE
)

PAT_MONTO = re.compile(
    r"(?P<prefix>\$|ARS|PESOS?)\s*(?P<value>\d{1,3}(?:\.\d{3})*(?:,\d{1,2})?|\d+(?:,\d{1,2})?)",
    re.IGNORECASE
)

PAT_ALIAS = re.compile(
    r"\bALIAS\s*(?:CBU|CVU)?\s*[:\-]?\s*(?P<value>[A-Z0-9][A-Z0-9\.\-]{4,40})\b",
    re.IGNORECASE
)


def extract_word_amount_before(text, money_start, window=180):
    """
    Intenta capturar el monto escrito en letras antes del monto numérico.
    Ejemplo:
    PESOS CIENTO OCHO MIL ... ($108.332,50)
    """
    before = text[max(0, money_start - window):money_start]
    before_clean = clean_context(before)

    m = re.search(
        r"(?i)(pesos?\s+[A-ZÁÉÍÓÚÑ\s]+(?:CON\s+(?:\d{1,2}/100|\w+\s+CENTAVOS?))?)\s*$",
        before_clean
    )

    return clean_context(m.group(1)) if m else None


def extract_regex_entities(text):
    entities = []

    # CBU / CVU
    for m in PAT_CBU_CVU.finditer(text):
        label = m.group("label").upper().replace(".", "")
        raw_value = m.group("value")
        norm = normalize_digits(raw_value)

        if len(norm) == 22:
            entities.append(
                entity_dict(
                    tipo=label,
                    valor=raw_value,
                    valor_normalizado=norm,
                    metodo="regex",
                    start=m.start("value"),
                    end=m.end("value"),
                    text=text,
                    confianza=0.97
                )
            )

    # CUIT / CUIL
    for m in PAT_CUIL_CUIT.finditer(text):
        label = m.group("label").upper().replace(".", "")
        raw_value = m.group("value")
        norm = normalize_digits(raw_value)

        if len(norm) == 11:
            entities.append(
                entity_dict(
                    tipo="CUIT_CUIL",
                    valor=raw_value,
                    valor_normalizado=norm,
                    metodo="regex",
                    start=m.start("value"),
                    end=m.end("value"),
                    text=text,
                    confianza=0.96,
                    extra={"subtipo": label}
                )
            )

    # DNI
    for m in PAT_DNI.finditer(text):
        raw_value = m.group("value")
        norm = normalize_digits(raw_value)

        if 7 <= len(norm) <= 8:
            entities.append(
                entity_dict(
                    tipo="DNI",
                    valor=raw_value,
                    valor_normalizado=norm,
                    metodo="regex",
                    start=m.start("value"),
                    end=m.end("value"),
                    text=text,
                    confianza=0.94
                )
            )

    # Alias
    for m in PAT_ALIAS.finditer(text):
        raw_value = m.group("value")

        entities.append(
            entity_dict(
                tipo="ALIAS",
                valor=raw_value,
                valor_normalizado=raw_value.upper(),
                metodo="regex",
                start=m.start("value"),
                end=m.end("value"),
                text=text,
                confianza=0.85
            )
        )

    # Montos
    for m in PAT_MONTO.finditer(text):
        raw_value = (m.group("prefix") + " " + m.group("value")).strip()
        norm = normalize_money_ar(raw_value)

        if norm is not None and norm >= MIN_MONTO_CANDIDATO:
            entities.append(
                entity_dict(
                    tipo="MONTO",
                    valor=raw_value,
                    valor_normalizado=norm,
                    metodo="regex_contextual",
                    start=m.start(),
                    end=m.end(),
                    text=text,
                    confianza=None,
                    extra={
                        "texto_monto": extract_word_amount_before(text, m.start())
                    }
                )
            )

    return entities

# Celda 6 — Ranking contextual para elegir el monto principal

Acá no usamos LLM.
La idea es extraer todos los montos candidatos y después elegir el más probable según el contexto.

In [ ]:
def score_monto(ent, text):
    """
    Puntúa un monto candidato según palabras cercanas.

    Queremos priorizar montos cerca de:
    - embargo
    - hasta cubrir
    - importe del crédito
    - capital reclamado
    - deberá depositar

    Y penalizar montos secundarios:
    - intereses y costas
    - con más la suma
    - prima facie
    """
    start = ent["span_inicio"]
    end = ent["span_fin"]

    before = text[max(0, start - 250):start].lower()
    after = text[end:min(len(text), end + 180)].lower()
    ctx = before + " " + after

    immediate_before = text[max(0, start - 100):start].lower()
    immediate_after = text[end:min(len(text), end + 120)].lower()

    score = 0

    if "$" in safe_str(ent.get("valor")):
        score += 2

    positive_strong = [
        "hasta cubrir",
        "importe del crédito",
        "importe del credito",
        "capital reclamado",
        "se ha decretado el embargo",
        "decretado el embargo",
        "decrétase embargo",
        "decretase embargo",
        "trabar embargo",
    ]

    positive_medium = [
        "deberá ser depositada",
        "debera ser depositada",
        "deberá depositar",
        "debera depositar",
        "sumas embargadas",
        "embargo sobre",
    ]

    for kw in positive_strong:
        if kw in ctx:
            score += 5

    for kw in positive_medium:
        if kw in ctx:
            score += 3

    # Penalización: montos de intereses/costas suelen ser secundarios
    if any(kw in immediate_before for kw in ["con más la suma", "con mas la suma"]):
        score -= 8

    if any(kw in immediate_after for kw in ["intereses y costas", "costas", "provisoriamente", "prima facie"]):
        score -= 6

    # Penalización para evitar confundir con datos personales o bancarios
    false_context = [
        "dni",
        "cuit",
        "cuil",
        "cbu",
        "cvu",
        "código de verificación",
        "codigo de verificacion",
    ]

    if any(kw in ctx for kw in false_context):
        score -= 4

    # Suele haber un bloque principal antes de "Los autos que ordenan..."
    # y después aparecen transcripciones anteriores que pueden repetir montos.
    first_quoted_orders = text.lower().find("los autos que ordenan")

    if first_quoted_orders != -1 and start < first_quoted_orders:
        score += 4

    if first_quoted_orders != -1 and start > first_quoted_orders:
        score -= 1

    if ent.get("texto_monto"):
        score += 1

    if ent.get("valor_normalizado", 0) > 0:
        score += 1

    return score


def choose_best_monto(entities, text):
    montos = [e for e in entities if e["tipo"] == "MONTO"]

    if not montos:
        return None

    for e in montos:
        e["score_contextual"] = score_monto(e, text)

    best = sorted(
        montos,
        key=lambda e: (e["score_contextual"], -e["span_inicio"]),
        reverse=True
    )[0]

    best["confianza"] = round(
        max(0.35, min(0.98, 0.55 + best["score_contextual"] * 0.035)),
        2
    )

    return best

# Celda 7 — Cargar GLiNER para personas

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

try:
    gliner_model = GLiNER.from_pretrained(MODEL_NAME)

    try:
        gliner_model.to(device)
    except Exception:
        pass

    print("Modelo GLiNER cargado:", MODEL_NAME)

except Exception as e:
    print("No se pudo cargar GLiNER.")
    print("Error:", e)
    gliner_model = None

Dispositivo: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

No se pudo cargar GLiNER.
Error: No config file found in /root/.cache/huggingface/hub/models--fastino--gliner2-privacy-filter-PII-multi/snapshots/e40819b10177c9b9cdea62a3ece5cfdebd921e01


=================================================================================
# Celda 7 actualizada — Cargar GLiNER anterior
=================================================================================

In [ ]:
# =========================
# CELDA 7 ACTUALIZADA
# Cargar modelo GLiNER que funcionó mejor para personas
# =========================

MODEL_NAME = "urchade/gliner_multi_pii-v1"

# Para OCR judicial conviene empezar bajo para no perder nombres.
# Si trae demasiadas personas falsas, probar 0.30 o 0.35.
GLINER_LABELS = ["person"]
GLINER_THRESHOLD = 0.25

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

try:
    gliner_model = GLiNER.from_pretrained(MODEL_NAME)

    try:
        gliner_model.to(device)
    except Exception:
        pass

    print("Modelo GLiNER cargado:", MODEL_NAME)
    print("Labels:", GLINER_LABELS)
    print("Threshold:", GLINER_THRESHOLD)

except Exception as e:
    print("No se pudo cargar GLiNER.")
    print("Error:", e)
    gliner_model = None

Dispositivo: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

Modelo GLiNER cargado: urchade/gliner_multi_pii-v1
Labels: ['person']
Threshold: 0.25


# Celda 8 — Extraer personas con GLiNER

In [ ]:
def extract_persons_gliner(text, model):
    if model is None:
        return []

    entities = []

    for offset, chunk in chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP):
        if not chunk.strip():
            continue

        try:
            preds = model.predict_entities(
                chunk,
                GLINER_LABELS,
                threshold=GLINER_THRESHOLD
            )
        except Exception as e:
            print("Error GLiNER en chunk:", e)
            continue

        for p in preds:
            label = safe_str(p.get("label", "")).lower()
            value = safe_str(p.get("text") or p.get("span") or "").strip()

            if label != "person":
                continue

            if len(value) < 3:
                continue

            local_start = p.get("start")
            local_end = p.get("end")

            if local_start is None or local_end is None:
                # Fallback por si el modelo no devuelve spans
                found = chunk.find(value)
                if found == -1:
                    continue
                local_start = found
                local_end = found + len(value)

            start = offset + int(local_start)
            end = offset + int(local_end)

            entities.append(
                entity_dict(
                    tipo="persona",
                    valor=value,
                    valor_normalizado=value.upper(),
                    metodo="gliner",
                    start=start,
                    end=end,
                    text=text,
                    confianza=round(float(p.get("score", 0)), 4)
                )
            )

    return deduplicate_entities(entities)

================================================================================
# Celda 8 actualizada — Extraer personas con urchade/gliner_multi_pii-v1

Esta celda es casi igual, pero la dejo un poco más robusta para nombres con OCR roto.


In [ ]:
# =========================
# CELDA 8 ACTUALIZADA
# Extracción de personas con GLiNER
# =========================

def is_valid_person_candidate(value):
    """
    Filtro simple para evitar basura OCR.
    No es perfecto, pero ayuda a limpiar resultados raros.
    """
    value = clean_context(value)

    if len(value) < 5:
        return False

    # Evitar candidatos que son casi todo números
    digits = sum(ch.isdigit() for ch in value)
    letters = sum(ch.isalpha() for ch in value)

    if letters < 3:
        return False

    if digits > letters:
        return False

    # Evitar urls, mails y textos técnicos
    bad_tokens = [
        "http",
        "www",
        ".gov",
        "notificaciones",
        "firma",
        "certificado",
        "descargar",
        "texto",
        "datos",
        "cuit",
        "cuil",
        "dni",
        "cbu",
        "cvu",
        "expediente",
        "oficio",
    ]

    low = value.lower()

    if any(tok in low for tok in bad_tokens):
        return False

    return True


def extract_persons_gliner(text, model):
    if model is None:
        return []

    entities = []

    for offset, chunk in chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP):
        if not chunk.strip():
            continue

        try:
            preds = model.predict_entities(
                chunk,
                GLINER_LABELS,
                threshold=GLINER_THRESHOLD
            )
        except Exception as e:
            print("Error GLiNER en chunk:", e)
            continue

        for p in preds:
            label = safe_str(p.get("label", "")).lower()
            value = safe_str(p.get("text") or p.get("span") or "").strip()

            if label != "person":
                continue

            if not is_valid_person_candidate(value):
                continue

            local_start = p.get("start")
            local_end = p.get("end")

            if local_start is None or local_end is None:
                found = chunk.find(value)
                if found == -1:
                    continue
                local_start = found
                local_end = found + len(value)

            start = offset + int(local_start)
            end = offset + int(local_end)

            entities.append(
                entity_dict(
                    tipo="persona",
                    valor=value,
                    valor_normalizado=value.upper(),
                    metodo="gliner_urchade",
                    start=start,
                    end=end,
                    text=text,
                    confianza=round(float(p.get("score", 0)), 4)
                )
            )

    return deduplicate_entities(entities)

# Celda 9 — Elegir persona embargada, cuenta y armar estructura JSON

In [ ]:
def score_persona_embargada(ent, text):
    start = ent["span_inicio"]
    end = ent["span_fin"]

    before = text[max(0, start - 260):start].lower()
    after = text[end:min(len(text), end + 160)].lower()
    ctx = before + " " + after

    score = float(ent.get("confianza") or 0)

    positive = [
        "parte demandada",
        "demandada en autos",
        "demandado en autos",
        "ejecutado",
        "ejecutada",
        "deudor",
        "deudora",
        " c/ ",
        "contra",
        "pertenezca al ejecutado",
    ]

    negative = [
        "juez",
        "jueza",
        "secretaria",
        "secretario",
        "firmado",
        "notificado",
        "certificado",
        "dr.",
        "dra.",
        "autoriz",
        "diligenciamiento",
        "gerente",
        "mercado libre",
    ]

    for kw in positive:
        if kw in ctx:
            score += 2

    for kw in negative:
        if kw in ctx:
            score -= 2

    return score


def choose_persona_embargada(entities, text):
    personas = [e for e in entities if e["tipo"] == "persona"]

    if not personas:
        return None

    for e in personas:
        e["score_rol_embargado"] = score_persona_embargada(e, text)

    best = sorted(
        personas,
        key=lambda e: (e["score_rol_embargado"], e.get("confianza") or 0),
        reverse=True
    )[0]

    return best


def nearest_entity(target, candidates, max_dist=350):
    if target is None or not candidates:
        return None

    t_start = target.get("span_inicio")
    t_end = target.get("span_fin")

    best = None
    best_dist = None

    for c in candidates:
        c_start = c.get("span_inicio")
        c_end = c.get("span_fin")

        if c_start is None or c_end is None:
            continue

        if c_start >= t_end:
            dist = c_start - t_end
        elif t_start >= c_end:
            dist = t_start - c_end
        else:
            dist = 0

        if best_dist is None or dist < best_dist:
            best = c
            best_dist = dist

    if best_dist is not None and best_dist <= max_dist:
        return best

    return None


def infer_bank_from_context(context):
    context = clean_context(context)

    m = re.search(
        r"\b(Banco\s+(?:de\s+la\s+)?[A-ZÁÉÍÓÚÑa-z\s\.]+?)(?:,|\.|\-|Sucursal|a nombre|$)",
        context,
        re.IGNORECASE
    )

    if m:
        return clean_context(m.group(1))

    return None


def choose_cuenta_deposito(entities):
    cuentas = [e for e in entities if e["tipo"] in ["CBU", "CVU"]]
    aliases = [e for e in entities if e["tipo"] == "ALIAS"]

    if not cuentas:
        return {
            "cbu": None,
            "cvu": None,
            "alias": aliases[0]["valor_normalizado"] if aliases else None,
            "banco": None,
            "titular": None,
            "metodo": "regex",
            "confianza": None,
            "contexto": None,
        }

    def score_cuenta(e):
        ctx = safe_str(e.get("contexto", "")).lower()
        score = 0

        for kw in [
            "deberá ser depositada",
            "debera ser depositada",
            "deberá depositar",
            "debera depositar",
            "cuenta judicial",
            "banco",
            "sucursal",
            "a nombre",
            "orden de este juzgado",
        ]:
            if kw in ctx:
                score += 2

        return score

    best = sorted(cuentas, key=score_cuenta, reverse=True)[0]

    return {
        "cbu": best["valor_normalizado"] if best["tipo"] == "CBU" else None,
        "cvu": best["valor_normalizado"] if best["tipo"] == "CVU" else None,
        "alias": aliases[0]["valor_normalizado"] if aliases else None,
        "banco": infer_bank_from_context(best.get("contexto")),
        "titular": None,
        "metodo": "regex",
        "confianza": best.get("confianza"),
        "contexto": best.get("contexto"),
    }


def build_persona_embargada(entities, text):
    persona = choose_persona_embargada(entities, text)

    dnis = [e for e in entities if e["tipo"] == "DNI"]
    cuils = [e for e in entities if e["tipo"] == "CUIT_CUIL"]

    dni = nearest_entity(persona, dnis, max_dist=400) if persona else None
    cuil_cuit = nearest_entity(persona, cuils, max_dist=450) if persona else None

    return {
        "nombre": persona["valor"] if persona else None,
        "dni": dni["valor_normalizado"] if dni else None,
        "cuil_cuit": cuil_cuit["valor_normalizado"] if cuil_cuit else None,
        "metodo": {
            "nombre": "gliner" if persona else None,
            "dni": "regex" if dni else None,
            "cuil_cuit": "regex" if cuil_cuit else None,
        },
        "confianza": persona.get("confianza") if persona else None,
        "contexto": persona.get("contexto") if persona else None,
    }


def build_monto_json(best_monto):
    if best_monto is None:
        return {
            "valor_raw": None,
            "valor_normalizado": None,
            "moneda": "ARS",
            "texto_monto": None,
            "contexto": None,
            "metodo": "regex_contextual",
            "confianza": None,
        }

    return {
        "valor_raw": best_monto.get("valor"),
        "valor_normalizado": best_monto.get("valor_normalizado"),
        "moneda": "ARS",
        "texto_monto": best_monto.get("texto_monto"),
        "contexto": best_monto.get("contexto"),
        "metodo": "regex_contextual",
        "confianza": best_monto.get("confianza"),
        "score_contextual": best_monto.get("score_contextual"),
    }


def process_embargo(row, numero_embargo):
    text = safe_str(row[text_col])

    doc_id = safe_str(row[id_col]) if id_col else f"embargo_{numero_embargo:03d}"
    nombre = safe_str(row[nombre_col]) if nombre_col else None
    clasificacion = safe_str(row[clasificacion_col]) if clasificacion_col else "embargo"

    regex_entities = extract_regex_entities(text)
    person_entities = extract_persons_gliner(text, gliner_model)

    entities = deduplicate_entities(regex_entities + person_entities)

    best_monto = choose_best_monto(entities, text)
    persona_embargada = build_persona_embargada(entities, text)
    cuenta_deposito = choose_cuenta_deposito(entities)

    counts = Counter(e["tipo"] for e in entities)

    motivos_revision = []

    if not persona_embargada.get("nombre"):
        motivos_revision.append("No se detectó persona embargada")

    if not persona_embargada.get("dni") and not persona_embargada.get("cuil_cuit"):
        motivos_revision.append("No se detectó DNI/CUIL/CUIT asociado a la persona")

    if not best_monto:
        motivos_revision.append("No se detectó monto principal")

    if not cuenta_deposito.get("cbu") and not cuenta_deposito.get("cvu") and not cuenta_deposito.get("alias"):
        motivos_revision.append("No se detectó cuenta de depósito")

    if best_monto and best_monto.get("confianza") is not None and best_monto.get("confianza") < 0.65:
        motivos_revision.append("Monto con baja confianza contextual")

    revision_manual = len(motivos_revision) > 0

    result = {
        "numero_embargo": int(numero_embargo),
        "id": doc_id,
        "nombre_archivo": nombre,
        "clasificacion": clasificacion,
        "texto_limpio": text,

        "persona_embargada": persona_embargada,
        "monto": build_monto_json(best_monto),
        "cuenta_deposito": cuenta_deposito,

        "cantidad_entidades_encontradas": len(entities),
        "estadisticas_entidades": {
            "total": len(entities),
            "por_tipo": dict(counts)
        },

        "entidades": entities,

        "revision_manual": revision_manual,
        "motivos_revision": motivos_revision,
    }

    return to_jsonable(result)

# Celda 9 actualizada — Mejorar elección de persona embargada

Te recomiendo actualizar esta también, porque ahora que GLiNER sí va a traer personas, hay que elegir mejor cuál es la embargada y no quedarse con juez, secretaria, abogado o firmante.

In [ ]:
# =========================
# CELDA 9 ACTUALIZADA
# Elegir persona embargada, cuenta y estructura JSON
# =========================

def score_persona_embargada(ent, text):
    start = ent["span_inicio"]
    end = ent["span_fin"]

    before = text[max(0, start - 320):start].lower()
    after = text[end:min(len(text), end + 220)].lower()
    ctx = before + " " + after

    score = float(ent.get("confianza") or 0)

    positive_strong = [
        "parte demandada",
        "demandada en autos",
        "demandado en autos",
        "el demandado",
        "la demandada",
        "ejecutado",
        "ejecutada",
        "deudor",
        "deudora",
        "embargado",
        "embargada",
        "titular del dni",
        "titular dni",
        "fondos que posea",
        "fondos que tenga",
        "cuentas que posea",
        "cuenta que posee",
        "mercado pago del sr",
        "mercado pago de la sra",
        "pertenezca al ejecutado",
        "pertenezca a la ejecutada",
    ]

    positive_medium = [
        " dni ",
        " d.n.i",
        " cuil",
        " cuit",
        "hasta cubrir",
        "por la suma",
        "trábese embargo",
        "trabese embargo",
        "trabase embargo",
        "trabar embargo",
        "proceder a embargar",
    ]

    negative_strong = [
        "juez",
        "jueza",
        "secretaria",
        "secretario",
        "auxiliar letrado",
        "firmado",
        "notificado",
        "certificado",
        "fecha de firma",
        "usuario conectado",
        "autorizado",
        "autorizada",
        "diligenciar",
        "diligenciamiento",
        "gerente",
        "responsable",
        "mercado libre",
        "mercado pago",
        "poder judicial",
    ]

    negative_medium = [
        "dr.",
        "dra.",
        "abogado",
        "abogada",
        "letrado",
        "letrada",
        "sello",
        "domicilio",
    ]

    for kw in positive_strong:
        if kw in ctx:
            score += 3

    for kw in positive_medium:
        if kw in ctx:
            score += 1.5

    for kw in negative_strong:
        if kw in ctx:
            score -= 3

    for kw in negative_medium:
        if kw in ctx:
            score -= 1.5

    # Si el nombre aparece cerca de un DNI/CUIL, suma bastante.
    nearby = text[max(0, start - 80):min(len(text), end + 120)].lower()
    if "dni" in nearby or "d.n.i" in nearby or "cuil" in nearby or "cuit" in nearby:
        score += 4

    return score


def choose_persona_embargada(entities, text):
    personas = [e for e in entities if e["tipo"] == "persona"]

    if not personas:
        return None

    for e in personas:
        e["score_rol_embargado"] = score_persona_embargada(e, text)

    best = sorted(
        personas,
        key=lambda e: (e["score_rol_embargado"], e.get("confianza") or 0),
        reverse=True
    )[0]

    return best


def nearest_entity(target, candidates, max_dist=450):
    if target is None or not candidates:
        return None

    t_start = target.get("span_inicio")
    t_end = target.get("span_fin")

    best = None
    best_dist = None

    for c in candidates:
        c_start = c.get("span_inicio")
        c_end = c.get("span_fin")

        if c_start is None or c_end is None:
            continue

        if c_start >= t_end:
            dist = c_start - t_end
        elif t_start >= c_end:
            dist = t_start - c_end
        else:
            dist = 0

        if best_dist is None or dist < best_dist:
            best = c
            best_dist = dist

    if best_dist is not None and best_dist <= max_dist:
        return best

    return None


def infer_bank_from_context(context):
    context = clean_context(context)

    m = re.search(
        r"\b(Banco\s+(?:de\s+la\s+)?[A-ZÁÉÍÓÚÑa-z\s\.]+?)(?:,|\.|\-|Sucursal|a nombre|$)",
        context,
        re.IGNORECASE
    )

    if m:
        return clean_context(m.group(1))

    return None


def choose_cuenta_deposito(entities):
    cuentas = [e for e in entities if e["tipo"] in ["CBU", "CVU"]]
    aliases = [e for e in entities if e["tipo"] == "ALIAS"]

    if not cuentas:
        return {
            "cbu": None,
            "cvu": None,
            "alias": aliases[0]["valor_normalizado"] if aliases else None,
            "banco": None,
            "titular": None,
            "metodo": "regex",
            "confianza": None,
            "contexto": None,
        }

    def score_cuenta(e):
        ctx = safe_str(e.get("contexto", "")).lower()
        score = 0

        for kw in [
            "deberá ser depositada",
            "debera ser depositada",
            "deberá depositar",
            "debera depositar",
            "cuenta judicial",
            "banco",
            "sucursal",
            "a nombre",
            "orden de este juzgado",
        ]:
            if kw in ctx:
                score += 2

        return score

    best = sorted(cuentas, key=score_cuenta, reverse=True)[0]

    return {
        "cbu": best["valor_normalizado"] if best["tipo"] == "CBU" else None,
        "cvu": best["valor_normalizado"] if best["tipo"] == "CVU" else None,
        "alias": aliases[0]["valor_normalizado"] if aliases else None,
        "banco": infer_bank_from_context(best.get("contexto")),
        "titular": None,
        "metodo": "regex",
        "confianza": best.get("confianza"),
        "contexto": best.get("contexto"),
    }


def build_persona_embargada(entities, text):
    persona = choose_persona_embargada(entities, text)

    dnis = [e for e in entities if e["tipo"] == "DNI"]
    cuils = [e for e in entities if e["tipo"] == "CUIT_CUIL"]

    dni = nearest_entity(persona, dnis, max_dist=500) if persona else None
    cuil_cuit = nearest_entity(persona, cuils, max_dist=550) if persona else None

    return {
        "nombre": persona["valor"] if persona else None,
        "dni": dni["valor_normalizado"] if dni else None,
        "cuil_cuit": cuil_cuit["valor_normalizado"] if cuil_cuit else None,
        "metodo": {
            "nombre": "gliner_urchade" if persona else None,
            "dni": "regex" if dni else None,
            "cuil_cuit": "regex" if cuil_cuit else None,
        },
        "confianza": persona.get("confianza") if persona else None,
        "score_rol_embargado": persona.get("score_rol_embargado") if persona else None,
        "contexto": persona.get("contexto") if persona else None,
    }


def build_monto_json(best_monto):
    if best_monto is None:
        return {
            "valor_raw": None,
            "valor_normalizado": None,
            "moneda": "ARS",
            "texto_monto": None,
            "contexto": None,
            "metodo": "regex_contextual",
            "confianza": None,
        }

    return {
        "valor_raw": best_monto.get("valor"),
        "valor_normalizado": best_monto.get("valor_normalizado"),
        "moneda": "ARS",
        "texto_monto": best_monto.get("texto_monto"),
        "contexto": best_monto.get("contexto"),
        "metodo": "regex_contextual",
        "confianza": best_monto.get("confianza"),
        "score_contextual": best_monto.get("score_contextual"),
    }


def process_embargo(row, numero_embargo):
    text = safe_str(row[text_col])

    doc_id = safe_str(row[id_col]) if id_col else f"embargo_{numero_embargo:03d}"
    nombre = safe_str(row[nombre_col]) if nombre_col else None
    clasificacion = safe_str(row[clasificacion_col]) if clasificacion_col else "embargo"

    regex_entities = extract_regex_entities(text)
    person_entities = extract_persons_gliner(text, gliner_model)

    entities = deduplicate_entities(regex_entities + person_entities)

    best_monto = choose_best_monto(entities, text)
    persona_embargada = build_persona_embargada(entities, text)
    cuenta_deposito = choose_cuenta_deposito(entities)

    counts = Counter(e["tipo"] for e in entities)

    motivos_revision = []

    if not persona_embargada.get("nombre"):
        motivos_revision.append("No se detectó persona embargada")

    if not persona_embargada.get("dni") and not persona_embargada.get("cuil_cuit"):
        motivos_revision.append("No se detectó DNI/CUIL/CUIT asociado a la persona")

    if not best_monto:
        motivos_revision.append("No se detectó monto principal")

    if not cuenta_deposito.get("cbu") and not cuenta_deposito.get("cvu") and not cuenta_deposito.get("alias"):
        motivos_revision.append("No se detectó cuenta de depósito")

    if best_monto and best_monto.get("confianza") is not None and best_monto.get("confianza") < 0.65:
        motivos_revision.append("Monto con baja confianza contextual")

    revision_manual = len(motivos_revision) > 0

    result = {
        "numero_embargo": int(numero_embargo),
        "id": doc_id,
        "nombre_archivo": nombre,
        "clasificacion": clasificacion,
        "texto_limpio": text,

        "persona_embargada": persona_embargada,
        "monto": build_monto_json(best_monto),
        "cuenta_deposito": cuenta_deposito,

        "cantidad_entidades_encontradas": len(entities),
        "estadisticas_entidades": {
            "total": len(entities),
            "por_tipo": dict(counts)
        },

        "entidades": entities,

        "revision_manual": revision_manual,
        "motivos_revision": motivos_revision,
    }

    return to_jsonable(result)

# Celda 10 — Ejecutar extracción sobre los 80 embargos

In [ ]:
resultados = []

for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df)), start=1):
    resultado = process_embargo(row, numero_embargo=i)
    resultados.append(resultado)

print("Embargos procesados:", len(resultados))
print("Ejemplo primer embargo:")
print(json.dumps(resultados[0], ensure_ascii=False, indent=2)[:2500])

  0%|          | 0/80 [00:00<?, ?it/s]

Embargos procesados: 80
Ejemplo primer embargo:
{
  "numero_embargo": 1,
  "id": "538118",
  "nombre_archivo": "Embargo - usuario",
  "clasificacion": "Embargo",
  "texto_limpio": "18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y\n\nin | / /\no 000800 000 vo)\n\nUsuafio conectado: THEILER Pablo Maximiliano - 203222152320 notr\n\n| Orgañismo: juzgado EN LO CIVIL Y COMERCIAL N0%16 - SA\n| Chrátula: o AS SA C/ MARQUEZ NA 02 Ln e eu CUTIVO\nNúmero de causa: s/prrse-2093 o\nTipo de notificación: Jeeps ELECTRONICO 4 paar\nDestinatarios: A Y\nFécha notificación: 20/2/2026 \" O.\nAlta a disponibilidad 19/2/2026 10:44:34\n| Fifmal digital: Firma valida\nFirmablo y Notificado por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026\n10:44.33\nFifmabo por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026\n\n2]¡BRANDARIZ Luciana Rocio. --- Certificado Correcto. Fecha de Firma:\n2:20p6 32:44.42 THEILER Pablo M

# Celda 10 actualizada — Reprocesar y sobrescribir resultados

In [ ]:
# =========================
# CELDA 10 ACTUALIZADA
# Reprocesar los 80 embargos
# =========================

resultados = []

for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df)), start=1):
    resultado = process_embargo(row, numero_embargo=i)
    resultados.append(resultado)

print("Embargos procesados:", len(resultados))

# Mini diagnóstico para verificar si ahora aparecen personas
total_personas = sum(
    doc["estadisticas_entidades"]["por_tipo"].get("persona", 0)
    for doc in resultados
)

embargos_con_persona = sum(
    1 for doc in resultados
    if doc["persona_embargada"].get("nombre")
)

print("Total entidades persona detectadas:", total_personas)
print("Embargos con persona_embargada detectada:", embargos_con_persona, "de", len(resultados))

print("\nEjemplo primer embargo:")
print(json.dumps(resultados[0], ensure_ascii=False, indent=2)[:3000])

  0%|          | 0/80 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 633 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 599 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 632 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 549 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/usr/local/lib/python3.12/dist-p

Embargos procesados: 80
Total entidades persona detectadas: 936
Embargos con persona_embargada detectada: 80 de 80

Ejemplo primer embargo:
{
  "numero_embargo": 1,
  "id": "538118",
  "nombre_archivo": "Embargo - usuario",
  "clasificacion": "Embargo",
  "texto_limpio": "18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y\n\nin | / /\no 000800 000 vo)\n\nUsuafio conectado: THEILER Pablo Maximiliano - 203222152320 notr\n\n| Orgañismo: juzgado EN LO CIVIL Y COMERCIAL N0%16 - SA\n| Chrátula: o AS SA C/ MARQUEZ NA 02 Ln e eu CUTIVO\nNúmero de causa: s/prrse-2093 o\nTipo de notificación: Jeeps ELECTRONICO 4 paar\nDestinatarios: A Y\nFécha notificación: 20/2/2026 \" O.\nAlta a disponibilidad 19/2/2026 10:44:34\n| Fifmal digital: Firma valida\nFirmablo y Notificado por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026\n10:44.33\nFifmabo por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026\n\n2]¡BRANDAR

# Celda 11 — Guardar JSON y CSV

Vamos a guardar:

1. embargos_extraccion_80.json
2. embargos_extraccion_resumen.csv
3. embargos_entidades_detectadas.csv

In [ ]:
json_path = OUTPUT_DIR / "embargos_extraccion_80.json"
summary_csv_path = OUTPUT_DIR / "embargos_extraccion_resumen.csv"
entities_csv_path = OUTPUT_DIR / "embargos_entidades_detectadas.csv"

# JSON completo
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)


# CSV resumen, una fila por embargo
summary_rows = []

for doc in resultados:
    persona = doc["persona_embargada"]
    monto = doc["monto"]
    cuenta = doc["cuenta_deposito"]
    stats = doc["estadisticas_entidades"]["por_tipo"]

    summary_rows.append({
        "numero_embargo": doc["numero_embargo"],
        "id": doc["id"],
        "nombre_archivo": doc["nombre_archivo"],
        "clasificacion": doc["clasificacion"],

        "persona_nombre": persona.get("nombre"),
        "persona_dni": persona.get("dni"),
        "persona_cuil_cuit": persona.get("cuil_cuit"),
        "persona_confianza": persona.get("confianza"),

        "monto_valor_raw": monto.get("valor_raw"),
        "monto_valor_normalizado": monto.get("valor_normalizado"),
        "monto_moneda": monto.get("moneda"),
        "monto_confianza": monto.get("confianza"),
        "monto_score_contextual": monto.get("score_contextual"),

        "cuenta_cbu": cuenta.get("cbu"),
        "cuenta_cvu": cuenta.get("cvu"),
        "cuenta_alias": cuenta.get("alias"),
        "cuenta_banco": cuenta.get("banco"),
        "cuenta_confianza": cuenta.get("confianza"),

        "cantidad_entidades_encontradas": doc["cantidad_entidades_encontradas"],
        "cant_personas": stats.get("persona", 0),
        "cant_dni": stats.get("DNI", 0),
        "cant_cuit_cuil": stats.get("CUIT_CUIL", 0),
        "cant_cbu": stats.get("CBU", 0),
        "cant_cvu": stats.get("CVU", 0),
        "cant_alias": stats.get("ALIAS", 0),
        "cant_montos": stats.get("MONTO", 0),

        "revision_manual": doc["revision_manual"],
        "motivos_revision": " | ".join(doc["motivos_revision"]),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")


# CSV largo, una fila por entidad encontrada
entity_rows = []

for doc in resultados:
    for n, ent in enumerate(doc["entidades"], start=1):
        entity_rows.append({
            "grupo_revision": doc["numero_embargo"],
            "numero_embargo": doc["numero_embargo"],
            "id": doc["id"],
            "nombre_archivo": doc["nombre_archivo"],
            "entidad_n": n,

            "tipo_entidad": ent.get("tipo"),
            "valor": ent.get("valor"),
            "valor_normalizado": ent.get("valor_normalizado"),
            "metodo": ent.get("metodo"),
            "confianza": ent.get("confianza"),
            "score_contextual": ent.get("score_contextual"),
            "span_inicio": ent.get("span_inicio"),
            "span_fin": ent.get("span_fin"),
            "contexto": ent.get("contexto"),

            "estado_revision": "pendiente",
            "valor_corregido": "",
            "observaciones": "",
        })

entities_df = pd.DataFrame(entity_rows)
entities_df.to_csv(entities_csv_path, index=False, encoding="utf-8-sig")

print("Archivos guardados:")
print(json_path)
print(summary_csv_path)
print(entities_csv_path)

summary_df.head()

Archivos guardados:
/content/salida_extraccion_embargos/embargos_extraccion_80.json
/content/salida_extraccion_embargos/embargos_extraccion_resumen.csv
/content/salida_extraccion_embargos/embargos_entidades_detectadas.csv


,numero_embargo,id,nombre_archivo,clasificacion,persona_nombre,persona_dni,persona_cuil_cuit,persona_confianza,monto_valor_raw,monto_valor_normalizado,...,cantidad_entidades_encontradas,cant_personas,cant_dni,cant_cuit_cuil,cant_cbu,cant_cvu,cant_alias,cant_montos,revision_manual,motivos_revision
0,1,538118,Embargo - usuario,Embargo,NAHUEL OSCAR MARQUEZ,None,None,0.9853,"$ 108.332,50",108332.50,...,29,17,3,1,1,0,0,7,True,No se detectó DNI/CUIL/CUIT asociado a la persona
1,2,538084,Embargo - usuario,Embargo,MAZA ARIAS MERCEDES FATIMA,32630518,None,0.7560,$ 42.000,42000.00,...,11,6,1,1,1,0,0,2,True,Monto con baja confianza contextual
2,3,495011,Embargo - usuario,Embargo,Gral San Martín,34436998,20344369985,0.8440,"$ 589.342,05",589342.05,...,24,14,2,2,2,0,1,3,False,
3,4,534526,Embargo - Empleado,Embargo,BRAVO LUIS ALBERTO,None,None,0.9806,None,NaN,...,13,11,1,0,1,0,0,0,True,No se detectó DNI/CUIL/CUIT asociado a la pers...
4,5,542996,Embargo - usuario,Embargo,STIO MARIA AMELIA,6551699,None,0.7938,$ 4.500,4500.00,...,14,7,2,1,1,0,0,3,True,Monto con baja confianza contextual


# Celda 12 — Crear Excel para revisión manual

Este Excel tiene:

* hoja Revision_Entidades: una fila por entidad;
* hoja Resumen_Embargos: una fila por embargo;
* columna auxiliar grupo_revision, de 1 a 80;
* color alternado por número de embargo;
* columna estado_revision con opciones;
* columnas para corrección manual.



In [ ]:
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.worksheet.datavalidation import DataValidation


excel_path = OUTPUT_DIR / "embargos_revision_manual.xlsx"


def write_df_to_sheet(ws, dataframe):
    for r in dataframe_to_rows(dataframe, index=False, header=True):
        ws.append(r)


def style_sheet(ws, title_fill="1F4E78"):
    header_fill = PatternFill("solid", fgColor=title_fill)
    header_font = Font(color="FFFFFF", bold=True)
    thin = Side(style="thin", color="D9E2F3")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # Header
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = border

    # Body
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.border = border
            cell.alignment = Alignment(vertical="top", wrap_text=True)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions


def apply_group_colors(ws, group_col_name="grupo_revision"):
    headers = {cell.value: cell.column for cell in ws[1]}

    if group_col_name not in headers:
        return

    group_col = headers[group_col_name]

    fill_odd = PatternFill("solid", fgColor="EAF3FF")
    fill_even = PatternFill("solid", fgColor="FFF2CC")
    fill_group_odd = PatternFill("solid", fgColor="9DC3E6")
    fill_group_even = PatternFill("solid", fgColor="FFD966")

    for row_idx in range(2, ws.max_row + 1):
        value = ws.cell(row=row_idx, column=group_col).value

        try:
            grupo = int(value)
        except Exception:
            grupo = 0

        fill = fill_odd if grupo % 2 == 1 else fill_even
        fill_group = fill_group_odd if grupo % 2 == 1 else fill_group_even

        for col_idx in range(1, ws.max_column + 1):
            ws.cell(row=row_idx, column=col_idx).fill = fill

        ws.cell(row=row_idx, column=group_col).fill = fill_group
        ws.cell(row=row_idx, column=group_col).font = Font(bold=True)


def set_column_widths(ws, widths_by_header):
    headers = {cell.value: cell.column_letter for cell in ws[1]}

    for header, width in widths_by_header.items():
        if header in headers:
            ws.column_dimensions[headers[header]].width = width


def add_revision_validation(ws):
    headers = {cell.value: cell.column_letter for cell in ws[1]}

    if "estado_revision" not in headers:
        return

    col = headers["estado_revision"]

    dv = DataValidation(
        type="list",
        formula1='"pendiente,bien,revisar,corregir"',
        allow_blank=False
    )

    ws.add_data_validation(dv)
    dv.add(f"{col}2:{col}{ws.max_row}")


# Crear workbook
wb = Workbook()

# Hoja 1: revisión manual
ws_review = wb.active
ws_review.title = "Revision_Entidades"

write_df_to_sheet(ws_review, entities_df)
style_sheet(ws_review, title_fill="17365D")
apply_group_colors(ws_review, group_col_name="grupo_revision")
add_revision_validation(ws_review)

set_column_widths(ws_review, {
    "grupo_revision": 14,
    "numero_embargo": 16,
    "id": 18,
    "nombre_archivo": 28,
    "entidad_n": 10,
    "tipo_entidad": 16,
    "valor": 32,
    "valor_normalizado": 24,
    "metodo": 18,
    "confianza": 12,
    "score_contextual": 16,
    "span_inicio": 12,
    "span_fin": 12,
    "contexto": 70,
    "estado_revision": 18,
    "valor_corregido": 28,
    "observaciones": 40,
})

# Hoja 2: resumen
ws_summary = wb.create_sheet("Resumen_Embargos")
write_df_to_sheet(ws_summary, summary_df)
style_sheet(ws_summary, title_fill="375623")

set_column_widths(ws_summary, {
    "numero_embargo": 16,
    "id": 18,
    "nombre_archivo": 30,
    "clasificacion": 18,
    "persona_nombre": 32,
    "persona_dni": 18,
    "persona_cuil_cuit": 20,
    "persona_confianza": 18,
    "monto_valor_raw": 18,
    "monto_valor_normalizado": 22,
    "monto_moneda": 14,
    "monto_confianza": 18,
    "monto_score_contextual": 22,
    "cuenta_cbu": 28,
    "cuenta_cvu": 28,
    "cuenta_alias": 24,
    "cuenta_banco": 34,
    "cuenta_confianza": 18,
    "cantidad_entidades_encontradas": 28,
    "motivos_revision": 60,
})

# Hoja 3: instrucciones
ws_info = wb.create_sheet("Instrucciones")
ws_info.append(["Campo", "Descripción"])
ws_info.append(["grupo_revision", "Celda auxiliar de revisión. Va de 1 a 80 y cambia de color por embargo."])
ws_info.append(["estado_revision", "Marcar pendiente, bien, revisar o corregir."])
ws_info.append(["valor_corregido", "Completar solo si el valor extraído está mal."])
ws_info.append(["observaciones", "Notas de revisión manual."])
ws_info.append(["tipo_entidad", "persona, DNI, CUIT_CUIL, CBU, CVU, ALIAS o MONTO."])
ws_info.append(["contexto", "Fragmento del texto donde se encontró la entidad."])

style_sheet(ws_info, title_fill="7030A0")
set_column_widths(ws_info, {
    "Campo": 24,
    "Descripción": 90,
})

wb.save(excel_path)

print("Excel de revisión manual guardado en:")
print(excel_path)

Excel de revisión manual guardado en:
/content/salida_extraccion_embargos/embargos_revision_manual.xlsx


# Celda 13 — Ver resultados y descargar archivos

In [ ]:
print("Archivos finales:")

for path in [
    json_path,
    summary_csv_path,
    entities_csv_path,
    excel_path
]:
    print(path)

Archivos finales:
/content/salida_extraccion_embargos/embargos_extraccion_80.json
/content/salida_extraccion_embargos/embargos_extraccion_resumen.csv
/content/salida_extraccion_embargos/embargos_entidades_detectadas.csv
/content/salida_extraccion_embargos/embargos_revision_manual.xlsx


In [ ]:
# Descargar desde Colab
from google.colab import files

files.download(str(json_path))
files.download(str(summary_csv_path))
files.download(str(entities_csv_path))
files.download(str(excel_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>